# Lesson 23: BERT and GPT — the two big Transformer families 🤝
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain how **BERT** is pretrained (masked words, next-sentence prediction) and why it gives **contextual** embeddings
2. **Fine-tune** a BERT-style model for sentiment classification
3. Explain how **GPT** models are pretrained, how they grew from GPT-1 to today, and how decoding settings change their output
4. Choose between BERT-style and GPT-style models for a task

**⏱️ Timing:** BERT pretraining 10' | contextual embeddings 10' | fine-tuning 20' | GPT 15' | compare 5'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 📖 vs ✍️ An analogy: the reader and the writer
In Lesson 8 you met **encoders** and **decoders**. The two most famous model families are built from them:
| | **BERT** (Google, 2018) | **GPT** (OpenAI, 2018 →) |
|---|---|---|
| Built from | Transformer **encoder** | Transformer **decoder** |
| Pretraining game | Fill in the **blanks** (sees words on both sides) | Predict the **next word** (sees only the left) |
| Like a… | **Reader** with a highlighter | **Writer** with a pen |
| Good at | Understanding: classifying, searching, extracting | Generating: chatting, writing, coding |
| Name | **B**idirectional **E**ncoder **R**epresentations from **T**ransformers | **G**enerative **P**re-trained **T**ransformer |

## Part 1: How BERT is pretrained 🎭
1. **Masked Language Modelling (MLM):** hide 15% of the words and guess them, using both sides
2. **Next Sentence Prediction (NSP):** given two sentences, did B really come after A? (Later research found this game matters less, and models like RoBERTa dropped it.)

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModel
import torch, numpy as np
device = 0 if torch.cuda.is_available() else -1

fill = pipeline("fill-mask", model="bert-base-uncased", device=device)
for s in ["The students opened their [MASK] and started the test.",
          "I went to the [MASK] to buy some bread.",
          "The capital of New Zealand is [MASK]."]:
    print(s)
    for r in fill(s, top_k=3):
        print(f"     {r['token_str']:12} {r['score']:.0%}")

## Part 2: Contextual embeddings — kiwi 🥝 ≠ kiwi 🐦
GloVe (Lesson 22) gave each word **one** vector. BERT reads the whole sentence first, so the same word gets a **different vector in each context**.

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def word_vector(sentence, word):
    inputs = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = bert(**inputs).last_hidden_state[0]
    tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
    return hidden[tokens.index(word)].numpy()

def cos(u, v): return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

s1 = "I deposited the money at the bank."
s2 = "The bank approved my loan."
s3 = "We had a picnic on the river bank."
v1, v2, v3 = (word_vector(s, "bank") for s in [s1, s2, s3])
print(f"money bank vs loan bank:  {cos(v1, v2):.2f}")
print(f"money bank vs river bank: {cos(v1, v3):.2f}   ← lower: BERT knows these are different 'banks'")

### 🧩 Exercise 1
Find another word with two meanings (*bat*, *match*, *train*, *spring*) and test it. Does BERT separate the meanings?

## Part 3: Fine-tuning BERT — pretrain once, specialise cheaply 🎯
Pretraining BERT took days on many processors. **Fine-tuning** adds a small classifier on top and trains for a few minutes on **your** task. We'll teach DistilBERT (a smaller, faster BERT — see Lesson 27) to judge movie reviews.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train = ds["train"].shuffle(seed=0).select(range(2000))      # small slice so it trains quickly
test = ds["test"].shuffle(seed=0).select(range(500))

name = "distilbert-base-uncased"                           # NOT yet trained for sentiment
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model.to(dev)

def collate(batch):
    enc = tok([b["text"] for b in batch], padding=True, truncation=True, max_length=64, return_tensors="pt")
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc

def accuracy():
    model.eval(); correct = 0
    with torch.no_grad():
        for batch in DataLoader(test, batch_size=64, collate_fn=collate):
            batch = {k: v.to(dev) for k, v in batch.items()}
            correct += (model(**batch).logits.argmax(1) == batch["labels"]).sum().item()
    return correct / len(test)

print(f"Accuracy BEFORE fine-tuning: {accuracy():.0%}   (about a coin flip)")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)    # tiny learning rate: just nudge the pretrained weights
model.train()
for epoch in range(2):
    for step, batch in enumerate(DataLoader(train, batch_size=16, shuffle=True, collate_fn=collate)):
        batch = {k: v.to(dev) for k, v in batch.items()}
        loss = model(**batch).loss               # Hugging Face models compute the loss for you
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if step % 40 == 0: print(f"epoch {epoch + 1} step {step:3}  loss {loss.item():.3f}")
    print(f"Accuracy after epoch {epoch + 1}: {accuracy():.0%}")

In [ ]:
model.eval()
for review in ["A heartfelt story with brilliant acting.", "Two hours of my life I will never get back.",
               "The plot was predictable but the jokes landed."]:
    enc = tok(review, return_tensors="pt").to(dev)
    p = torch.softmax(model(**enc).logits, dim=1)[0, 1].item()
    print(f"{p:5.0%} positive | {review}")

### 🧩 Exercise 2
1. Use only `range(200)` training reviews instead of 2000. How much does accuracy drop?
2. Why do we use a **tiny** learning rate (3e-5) for fine-tuning? (We don't want to destroy what BERT already knows.)

## Part 4: GPT — the writer ✍️
### From GPT-1 to today
| Model | Year | Parameters | What changed |
|---|---|---|---|
| GPT-1 | 2018 | 117 million | Showed "pretrain then fine-tune" works for a decoder |
| GPT-2 | 2019 | 1.5 billion | Wrote surprisingly fluent text; released in stages over misuse worries |
| GPT-3 | 2020 | 175 billion | **Few-shot learning** from examples in the prompt, no fine-tuning |
| ChatGPT (GPT-3.5) | 2022 | not published | Fine-tuned to follow instructions + **RLHF** (Lesson 27) |
| GPT-4 and later | 2023 → | not published | Images as input, stronger reasoning, longer context |

Many other companies build GPT-style (decoder-only) models too: Anthropic's Claude, Google's Gemini, Meta's LLaMA (Lesson 25). Let's run the original GPT-2 (small, 124M parameters).

In [ ]:
gpt2 = pipeline("text-generation", model="gpt2", device=device)
prompt = "In a surprising discovery, scientists found a group of penguins living in"
torch.manual_seed(0)
print("GREEDY (always the top word):\n ", gpt2(prompt, max_new_tokens=40, do_sample=False)[0]["generated_text"], "\n")
print("SAMPLING (temperature 0.9, top-p 0.95):\n ", gpt2(prompt, max_new_tokens=40, do_sample=True, temperature=0.9, top_p=0.95)[0]["generated_text"])

Greedy decoding often **loops** and repeats itself. Sampling adds variety (Lesson 24 explains temperature, top-k and top-p in detail).

### 🧩 Exercise 3: Base model vs chat model
GPT-2 is a **base model**: it only continues text, it doesn't follow instructions. Try the prompt `"Explain photosynthesis to a 10 year old:"`. Then compare with an **instruction-tuned** model:

In [ ]:
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=device)
q = "Explain photosynthesis to a 10 year old in 2 sentences."
print("BASE GPT-2: ", gpt2(q, max_new_tokens=40, do_sample=True)[0]["generated_text"], "\n")
print("CHAT MODEL: ", chat([{"role": "user", "content": q}], max_new_tokens=60)[0]["generated_text"][-1]["content"])

## Part 5: Which family for which job? 🧰
| Task | Better choice | Why |
|---|---|---|
| Spam filter for 1 million emails a day | BERT-style | Small, fast, cheap; just needs a label |
| Homework-help chatbot | GPT-style | Needs to write answers |
| Search engine for school documents | BERT-style (embeddings) | Compare meanings quickly |
| Write a story from a prompt | GPT-style | Generation |

### 🚀 Challenge
Take the fine-tuned sentiment model and test it on **tweets with slang and emoji**. Where does it fail? How would you fix it? (Hint: what data was it fine-tuned on?)

## 📌 Summary
- **BERT** = encoder, pretrained by filling blanks → great **contextual** embeddings and classifiers; easy to **fine-tune**
- **GPT** = decoder, pretrained by predicting the next word → great at **generating**; grew from 117M to hundreds of billions of parameters
- **Base** models continue text; **instruction-tuned** chat models follow requests